In [ ]:
#Anaconda 2.4
#Python 3.7 -- 3.6-3.9
#tensorflow_gpu-2.6.0 esta instala keras 2.6.0
#Compilador: MSVC 2019
#Construir herramientas: bazel 3.7.2
#cuDNN: 8.1 para CUDA 11.2
#Kit de herramientas CUDA: 11.2

In [ ]:
# ==============================================================
# Verificar si TensorFlow usa GPU o CPU + prueba rápida de rendimiento
# Compatible con TF 2.10.x en Windows
# ==============================================================

import time
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Built with CUDA?:", tf.test.is_built_with_cuda())

# Lista de GPUs físicas detectadas
gpus = tf.config.list_physical_devices('GPU')
print("GPUs físicas detectadas:", gpus)

# Habilita memory growth (evita que TF reserve toda la VRAM al inicio)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print("GPUs lógicas:", logical_gpus)
    except Exception as e:
        print("Aviso al configurar memory_growth:", e)
else:
    print("No se detectaron GPUs físicas (usará CPU).")

# Mensaje simple
print("\n== Interpretación ==")
if gpus:
    print("✅ TensorFlow ve al menos 1 GPU y podrá usarla.")
else:
    print("⚙️ TensorFlow está ejecutándose en CPU.")

# --------------------------------------------------------------
# Prueba rápida de rendimiento: multiplicación de matrices grande
# (si hay GPU debería ser notoriamente más rápida)
# --------------------------------------------------------------
def bench_matmul(dev):
    # Usa un tamaño relativamente grande para notar diferencia
    n = 2000
    with tf.device(dev):
        a = tf.random.normal([n, n])
        b = tf.random.normal([n, n])
        # "warmup"
        _ = tf.matmul(a, b)
        tf.experimental.numpy.random.seed(0)
        start = time.time()
        _ = tf.matmul(a, b)
        elapsed = time.time() - start
    return elapsed

print("\n== Benchmark matmul (ms, aprox) ==")
try:
    cpu_t = bench_matmul("/CPU:0")
    print(f"CPU: {cpu_t*1000:.0f} ms")
except Exception as e:
    print("Benchmark CPU falló:", e)

if gpus:
    try:
        gpu_t = bench_matmul("/GPU:0")
        print(f"GPU: {gpu_t*1000:.0f} ms")
        speedup = (cpu_t / gpu_t) if 'cpu_t' in locals() else None
        if speedup:
            print(f"Aceleración GPU vs CPU: x{speedup:.1f}")
    except Exception as e:
        print("Benchmark GPU falló:", e)
else:
    print("Sin GPU, no se ejecuta el benchmark en /GPU:0.")

# --------------------------------------------------------------
# Prueba de colocación explícita (op en GPU si existe)
# --------------------------------------------------------------
print("\n== Colocación de dispositivo para una operación simple ==")
try:
    with tf.device("/GPU:0"):
        x = tf.constant([1.0, 2.0, 3.0])
        y = tf.square(x)
    print("Operación ejecutada en /GPU:0 (si no hay error).")
except Exception as e:
    print("No se pudo ejecutar en /GPU:0:", e)


¿Qué es este modelo?
-Es una red neuronal convolucional (CNN) entrenada para clasificar imágenes en 10 deportes (≈ 77.128 imágenes). 
-Las CNN aprenden filtros (kernels) que detectan bordes, texturas, patrones y, en capas profundas, 
estructuras más complejas (balones, canchas, uniformes, etc.), para decidir la clase.

Características 
-Entrada: imágenes RGB (recomiendo normalizarlas a float32 y redimensionarlas a un tamaño uniforme, p. ej. 128×128×3).
-Arquitectura: varias capas Conv2D + BatchNorm + ReLU/LeakyReLU + MaxPooling, seguidas de capas densas finales.
-Pérdida: CategoricalCrossentropy (multiclase).
-Optimizador: Adagrad (lo usas), aunque en visión suelen funcionar muy bien Adam y SGD con momentum.
-Salida: softmax con 10 probabilidades (una por clase).

¿Qué es capaz de hacer?
-Clasificar nuevas imágenes en una de las 10 categorías de deporte con una probabilidad asociada.
-Generalizar a imágenes no vistas si el dataset es variado (ángulos, iluminación, fondos, cámaras) 
-y el entrenamiento está bien regularizado.

Métricas recomendadas
-Accuracy global.
-Matriz de confusión.

Precision, 
-Recall, 
-F1 por clase 
-macro‑promedios (macro y weighted).
-ROC‑AUC macro y PR‑AUC macro (one‑vs‑rest) si quieres análisis fino.

Curvas de aprendizaje (loss/accuracy train vs. val).

In [ ]:
import sys, subprocess

def pipi(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir"] + pkgs)

# 1) pip estable
pipi(["pip==24.0"])

# 2) protobuf compatible con TF 2.10.x
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "protobuf"])
pipi(["protobuf==3.20.3"])

# 3) dependencias que TF 2.10.x espera
pipi(["typing-extensions==4.5.0", "wrapt==1.14.1", "gast==0.4.0", "h5py==3.8.*"])

# 4) NumPy compatible (si necesitas cambiarlo desde el cuaderno)
# OJO: en conda es mejor instalar numpy con conda, pero si no puedes:
# pipi(["numpy==1.23.5"])

# 5) (Opcional) Reinstalar TF 2.10.1 si tienes otra versión
# pipi(["tensorflow==2.10.1"])

print("Listo. Reinicia el kernel y vuelve a importar tensorflow.")


In [ ]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

In [ ]:
import tensorflow as tf
print("TF:", tf.__version__)
print("Built with CUDA?", tf.test.is_built_with_cuda())
print("GPUs:", tf.config.list_physical_devices('GPU'))


In [ ]:
# ==============================================================
# CELDA DE VERIFICACIÓN DE DISPOSITIVO (GPU / CPU)
# ==============================================================
import tensorflow as tf
from tensorflow.python.client import device_lib

print("TensorFlow versión:", tf.__version__)
print("Built with CUDA?:", tf.test.is_built_with_cuda())
print("Compilado con cuDNN?:", tf.test.is_built_with_gpu_support())
print("GPUs detectadas:", tf.config.list_physical_devices('GPU'))
print("Dispositivos disponibles en el sistema:")
print(device_lib.list_local_devices())

# Interpretación sencilla
if tf.config.list_physical_devices('GPU'):
    print("\n✅ TensorFlow está usando GPU (procesamiento acelerado por CUDA).")
else:
    print("\n⚙️ TensorFlow está ejecutándose solo en CPU (sin aceleración por GPU).")


In [1]:
#Importar Librerías¶
import numpy as np
import os
import re
import matplotlib.pyplot as plt
%matplotlib inline
#from sklearn.model_selection import train_test_split
#from sklearn.metrics import classification_report
#conda install -c conda-forge matplotlib
import pandas as pd
from sklearn.model_selection import train_test_split
#from keras.utils import to_categorical
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
#from keras.layers.convolutional import Conv2D
from keras.layers import LeakyReLU
from tensorflow.keras.layers import MaxPooling2D
from keras.layers import Dropout
from keras.layers import Flatten
from keras.layers import Dense

ImportError: cannot import name 'defun_with_attributes' from 'tensorflow.python.eager.function' (C:\Users\jrmb8\anaconda3\envs\DEPORTES_CNN\lib\site-packages\tensorflow\python\eager\function.py)

In [2]:
from tensorflow.keras import activations
import tensorflow as tf
import numpy as np
import os
import re
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Dense, Dropout, Flatten, Input, Conv2D, MaxPooling2D, BatchNormalization, LeakyReLU
#from keras.layers.advanced_activations import LeakyReLU
#from tensorflow.keras.layers import backend as K
from tensorflow.keras.utils import plot_model
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, Masking, TimeDistributed
from tensorflow. keras.utils import plot_model

#para correfir u eror graphics 
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input
#tf.compat.v1.get_default_graph ()
import numpy as np
#from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, LeakyReLU, ZeroPadding2D, UpSampling2D
#from keras.layers.merge import add, concatenate
from tensorflow.keras.models import Model
import struct
import cv2


ImportError: cannot import name 'defun_with_attributes' from 'tensorflow.python.eager.function' (C:\Users\jrmb8\anaconda3\envs\DEPORTES_CNN\lib\site-packages\tensorflow\python\eager\function.py)

In [ ]:
#Cargar set de Imágenes¶ son 77128 imagenes de 10 deportes

dirname = os.path.join(os.getcwd(), 'sportimages')
imgpath = dirname + os.sep 

images = []
directories = []
dircount = []
prevRoot=''
cant=0

print("leyendo imagenes de ",imgpath)

for root, dirnames, filenames in os.walk(imgpath):
    for filename in filenames:
        if re.search("\.(jpg|jpeg|png|bmp|tiff)$", filename):
            cant=cant+1
            filepath = os.path.join(root, filename)
            image = plt.imread(filepath)
            images.append(image)
            b = "Leyendo..." + str(cant)
            print (b, end="\r")
            if prevRoot !=root:
                print(root, cant)
                prevRoot=root
                directories.append(root)
                dircount.append(cant)
                cant=0
dircount.append(cant)

dircount = dircount[1:]
dircount[0]=dircount[0]+1
print('Directorios leidos:',len(directories))
print("Imagenes en cada directorio", dircount)
print('suma Total de imagenes en subdirs:',sum(dircount))

In [ ]:
#Creamos las etiquetas
labels=[]
indice=0
for cantidad in dircount:
    for i in range(cantidad):
        labels.append(indice)
    indice=indice+1
print("Cantidad etiquetas creadas: ",len(labels))

In [ ]:
deportes=[]
indice=0
for directorio in directories:
    name = directorio.split(os.sep)
    print(indice , name[len(name)-1])
    deportes.append(name[len(name)-1])
    indice=indice+1

In [ ]:
y = np.array(labels)
X = np.array(images, dtype=np.uint8) #convierto de lista a numpy
#X = np.array(images[0], dtype=np.uint8) #convierto de lista a numpy
# Encuentra las etiquetas de lo valores de entrenamiento
classes = np.unique(y)
nClasses = len(classes)
print('Total number of outputs : ', nClasses)
print('Output classes : ', classes)

print('Valor de y: ',y)
print('Valor de X: ',X)

In [ ]:
#Creamos Sets de Entrenamiento y Test
train_X,test_X,train_Y,test_Y = train_test_split(X,y,test_size=0.2)
print('Training data shape : ', train_X.shape, train_Y.shape)
print('Testing data shape : ', test_X.shape, test_Y.shape)

In [ ]:
plt.figure(figsize=[5,5])

# Display the first image in training data
plt.subplot(121)
plt.imshow(train_X[0,:,:], cmap='gray')
plt.title("Ground Truth : {}".format(train_Y[0]))

# Display the first image in testing data
plt.subplot(122)
plt.imshow(test_X[0,:,:], cmap='gray')
plt.title("Ground Truth : {}".format(test_Y[0]))

In [ ]:
#Preprocesamos las imagenes
train_X = train_X.astype('float32')
test_X = test_X.astype('float32')
train_X = train_X / 255.
test_X = test_X / 255.

In [ ]:
#Hacemos el One-hot Encoding para la red
# Change the labels from categorical to one-hot encoding
# Cambiar las etiquetas de codificación categórica a codificación unívoca
train_Y_one_hot = to_categorical(train_Y)
test_Y_one_hot = to_categorical(test_Y)

# Display the change for category label using one-hot encoding
print('Original label:', train_Y[0])
print('After conversion to one-hot:', train_Y_one_hot[0])

In [ ]:
#Creamos el Set de Entrenamiento y Validación
#Mezclar todo y crear los grupos de entrenamiento y testing
train_X,valid_X,train_label,valid_label = train_test_split(train_X, train_Y_one_hot, test_size=0.2, random_state=13)

In [ ]:
print(train_X.shape,valid_X.shape,train_label.shape,valid_label.shape)


In [ ]:
#Creamos el modelo de CNN
#declaramos variables con los parámetros de configuración de la red
INIT_LR = 1e-3 # Valor inicial de learning rate. El valor 1e-3 corresponde con 0.001
epochs = 70 # Cantidad de iteraciones completas al conjunto de imagenes de entrenamiento
batch_size = 64 # cantidad de imágenes que se toman a la vez en memoria


In [ ]:
#tf.compat.v1.disable_eager_execution()
#print(tf.compat.v1.get_default_graph())
#error module 'tensorflow' no tiene atributo 'reset_default_graph'
#tf.compat.v1.GraphDef()   # -> en lugar de tf.GraphDef()   # 
#tf.compat.v2.io.gfile.GFile()   # -> instead of tf.gfile.GFile()
#Import keras.<something>.<something>
# Configurar PATH for Python 3.6
# La version original esta guardada en .bash_profile.pysave
#conda install matplotlib

sport_model = Sequential()
sport_model.add(Conv2D(32, kernel_size=(3, 3),activation='linear',padding='same',input_shape=(21,28,3)))
sport_model.add(LeakyReLU(alpha=0.1))
sport_model.add(MaxPooling2D((2, 2),padding='same'))
sport_model.add(Dropout(0.5))
sport_model.add(Flatten())
sport_model.add(Dense(32, activation='linear'))
sport_model.add(LeakyReLU(alpha=0.1))
sport_model.add(Dropout(0.5))
sport_model.add(Dense(nClasses, activation='softmax'))

In [ ]:
sport_model.summary()

In [ ]:
from tensorflow import keras
import tensorflow as tf
#import keras
#model = keras.models.load_model('my_model.h5', custom_objects={'tf': tf})

sport_model.compile(loss=keras.losses.categorical_crossentropy, 
       optimizer=tf.keras.optimizers.Adagrad(learning_rate=INIT_LR, decay=INIT_LR / 100),metrics=['accuracy'])

#from tensorflow.keras.optimizers import Adagrad
#opt=Adagrad(lr=0.0001, decay=1e-6)
#sport_model.compile(optimizer=opt,loss=keras.losses.categorical_crossentropy,metrics=['accuracy'])



In [ ]:
# este paso puede tomar varios minutos, dependiendo de tu ordenador, cpu y memoria ram libre
# como ejemplo, en mi Macbook pro tarda 4 minutos
#sport_train = sport_model.fit(train_X, train_label, 
#batch_size=batch_size,epochs=epochs,verbose=1,validation_data=(valid_X, valid_label))

from tensorflow.keras.callbacks import CSVLogger

# Callback que guarda el historial de entrenamiento en un CSV
csv_logger = CSVLogger("history_log.csv", separator=",", append=False)

# Este paso puede tardar varios minutos dependiendo de tu equipo
sport_train = sport_model.fit(
    train_X, train_label,
    batch_size=batch_size,
    epochs=epochs,
    verbose=1,
    validation_data=(valid_X, valid_label),
    callbacks=[csv_logger]   # <<--- aquí se añade
)


In [ ]:
# Guardar el modelo entrenado
sport_model.save("sports_mnist2.h5")

# Cargarlo de nuevo más adelante
from tensorflow import keras
new_model = keras.models.load_model("sports_mnist2.h5")

# Verificar que carga correctamente
new_model.summary()


In [ ]:
# Formato HDF5
sport_model.save("sports_cnn.h5")
loaded = tf.keras.models.load_model("sports_cnn.h5")

# o SavedModel (carpeta)
sport_model.save("sports_cnn_savedmodel")  # carpeta
loaded = tf.keras.models.load_model("sports_cnn_savedmodel")

In [ ]:
# === IMPORTS (una sola vez al inicio del notebook) ===
import tensorflow as tf
from tensorflow.keras.callbacks import CSVLogger
import pandas as pd

# === CALLBACK: guardará por época en history_log.csv ===
csv_logger = CSVLogger("history_log.csv", separator=",", append=False)


In [ ]:
import pandas as pd
def inspect_history_csv(csv_path="history_log.csv"):
    try:
        hist = pd.read_csv(csv_path)
        print("Columnas:", list(hist.columns))
        display(hist.head())
    except FileNotFoundError:
        print(f"No se encontró {csv_path}. Debes entrenar alguna vez con CSVLogger para generarlo.")


In [ ]:
#import pandas as pd
#import matplotlib.pyplot as plt

#def plot_history_from_csv(csv_path="history_log.csv", save_png=False):
#    hist = pd.read_csv(csv_path)

    # curva de pérdida
    #fig, ax = plt.subplots(figsize=(7,5))
    #if "loss" in hist.columns:
    #    ax.plot(hist["loss"], label="train_loss")
    #if "val_loss" in hist.columns:
    #    ax.plot(hist["val_loss"], label="val_loss")
    #ax.set_xlabel("Epochs"); ax.set_ylabel("Loss"); ax.set_title("Curva de pérdida")
    #ax.legend(); plt.tight_layout()
    #if save_png: plt.savefig("loss_curve.png", dpi=150)
    #plt.show()

    # curva de accuracy
    #fig, ax = plt.subplots(figsize=(7,5))
    #acc_col = "accuracy" if "accuracy" in hist.columns else ("acc" if "acc" in hist.columns else None)
    #val_acc_col = "val_accuracy" if "val_accuracy" in hist.columns else ("val_acc" if "val_acc" in hist.columns else None)

    #if acc_col:     ax.plot(hist[acc_col], label="train_acc")
    #if val_acc_col: ax.plot(hist[val_acc_col], label="val_acc")
    #ax.set_xlabel("Epochs"); ax.set_ylabel("Accuracy"); ax.set_title("Curva de accuracy")
    #ax.legend(); plt.tight_layout()
    #if save_png: plt.savefig("accuracy_curve.png", dpi=150)
    #plt.show()

# Uso (sin re-entrenar, siempre que ya exista history_log.csv)
#plot_history_from_csv("history_log.csv")

In [ ]:
#Evaluamos la red
test_eval = sport_model.evaluate(test_X, test_Y_one_hot, verbose=1)

In [ ]:
print('Test loss:', test_eval[0])
print('Test accuracy:', test_eval[1])

In [ ]:
metrics=["accuracy"]
#history = model.fit()
accuracy = sport_train.history['accuracy']
val_accuracy = sport_train.history['val_accuracy']
loss = sport_train.history['loss']
val_loss = sport_train.history['val_loss']
epochs = range(len(accuracy))
plt.plot(epochs, accuracy, 'bo', label='Training accuracy')
plt.plot(epochs, val_accuracy, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.legend()
plt.figure()
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()

In [ ]:
predicted_classes2 = sport_model.predict(test_X)

In [ ]:
predicted_classes=[]
for predicted_sport in predicted_classes2:
    predicted_classes.append(predicted_sport.tolist().index(max(predicted_sport)))
predicted_classes=np.array(predicted_classes)

In [ ]:
predicted_classes.shape, test_Y.shape

In [ ]:
#Aprendamos de los errores: Qué mejorar
correct = np.where(predicted_classes==test_Y)[0]
print("Found %d correct labels" % len(correct))
for i, correct in enumerate(correct[0:9]):
    plt.subplot(3,3,i+1)
    plt.imshow(test_X[correct].reshape(21,28,3), cmap='gray', interpolation='none')
    plt.title("{}, {}".format(deportes[predicted_classes[correct]],
                                                    deportes[test_Y[correct]]))

    plt.tight_layout()

In [ ]:
incorrect = np.where(predicted_classes!=test_Y)[0]
print("Found %d incorrect labels" % len(incorrect))
for i, incorrect in enumerate(incorrect[0:9]):
    plt.subplot(3,3,i+1)
    plt.imshow(test_X[incorrect].reshape(21,28,3), cmap='gray', interpolation='none')
    plt.title("{}, {}".format(deportes[predicted_classes[incorrect]],
                                                    deportes[test_Y[incorrect]]))
    plt.tight_layout()



In [ ]:
target_names = ["Class {}".format(i) for i in range(nClasses)]
print(classification_report(test_Y, predicted_classes, target_names=target_names))

In [ ]:
#prueba del modelo
#Prediccion de una nueva imagen¶
from skimage.transform import resize

images=[]
# se indica la ruta desde la cual esta l aimagen para probar el modelo
# En este caso 

filenames = ['../CLASIF_DEPORTES/test/deporte1.jpg']

for filepath in filenames:
    image = plt.imread(filepath,0)
    image_resized = resize(image, (21, 28),anti_aliasing=True,clip=False,preserve_range=True)
    images.append(image_resized)

X = np.array(images, dtype=np.uint8) #convierto de lista a numpy
test_X = X.astype('float32')
test_X = test_X / 255.

predicted_classes = sport_model.predict(test_X)

for i, img_tagged in enumerate(predicted_classes):
    print(filenames[i], deportes[img_tagged.tolist().index(max(img_tagged))])

In [ ]:
#prueba del modelo
#Prediccion de una nueva imagen¶
from skimage.transform import resize

images=[]
# se indica la ruta desde la cual esta l aimagen para probar el modelo
# En este caso 

filenames = ['../CLASIF_DEPORTES/test/deporte2.jpg']

for filepath in filenames:
    image = plt.imread(filepath,0)
    image_resized = resize(image, (21, 28),anti_aliasing=True,clip=False,preserve_range=True)
    images.append(image_resized)

X = np.array(images, dtype=np.uint8) #convierto de lista a numpy
test_X = X.astype('float32')
test_X = test_X / 255.

predicted_classes = sport_model.predict(test_X)

for i, img_tagged in enumerate(predicted_classes):
    print(filenames[i], deportes[img_tagged.tolist().index(max(img_tagged))])


In [ ]:
#prueba del modelo
#Prediccion de una nueva imagen¶
from skimage.transform import resize

images=[]
# se indica la ruta desde la cual esta l aimagen para probar el modelo
# En este caso 

filenames = ['../CLASIF_DEPORTES/test/iamgen21.jpg']

for filepath in filenames:
    image = plt.imread(filepath,0)
    image_resized = resize(image, (21, 28),anti_aliasing=True,clip=False,preserve_range=True)
    images.append(image_resized)

X = np.array(images, dtype=np.uint8) #convierto de lista a numpy
test_X = X.astype('float32')
test_X = test_X / 255.

predicted_classes = sport_model.predict(test_X)

for i, img_tagged in enumerate(predicted_classes):
    print(filenames[i], deportes[img_tagged.tolist().index(max(img_tagged))])


In [ ]:
#prueba del modelo
#Prediccion de una nueva imagen¶
from skimage.transform import resize

images=[]
# se indica la ruta desde la cual esta l aimagen para probar el modelo
# En este caso 

filenames = ['../CLASIF_DEPORTES/test/imagen3.jpg']

for filepath in filenames:
    image = plt.imread(filepath,0)
    image_resized = resize(image, (21, 28),anti_aliasing=True,clip=False,preserve_range=True)
    images.append(image_resized)

X = np.array(images, dtype=np.uint8) #convierto de lista a numpy
test_X = X.astype('float32')
test_X = test_X / 255.

predicted_classes = sport_model.predict(test_X)

for i, img_tagged in enumerate(predicted_classes):
    print(filenames[i], deportes[img_tagged.tolist().index(max(img_tagged))])

In [ ]:
#prueba del modelo
#Prediccion de una nueva imagen¶
from skimage.transform import resize

images=[]
# se indica la ruta desde la cual esta l aimagen para probar el modelo
# En este caso 

filenames = ['../CLASIF_DEPORTES/test/imagen16.jpg']

for filepath in filenames:
    image = plt.imread(filepath,0)
    image_resized = resize(image, (21, 28),anti_aliasing=True,clip=False,preserve_range=True)
    images.append(image_resized)

X = np.array(images, dtype=np.uint8) #convierto de lista a numpy
test_X = X.astype('float32')
test_X = test_X / 255.

predicted_classes = sport_model.predict(test_X)

for i, img_tagged in enumerate(predicted_classes):
    print(filenames[i], deportes[img_tagged.tolist().index(max(img_tagged))])

In [ ]:
#prueba del modelo
#Prediccion de una nueva imagen¶
from skimage.transform import resize

images=[]
# se indica la ruta desde la cual esta l aimagen para probar el modelo
# En este caso 

filenames = ['../CLASIF_DEPORTES/test/imagen14.jpg']

for filepath in filenames:
    image = plt.imread(filepath,0)
    image_resized = resize(image, (21, 28),anti_aliasing=True,clip=False,preserve_range=True)
    images.append(image_resized)

X = np.array(images, dtype=np.uint8) #convierto de lista a numpy
test_X = X.astype('float32')
test_X = test_X / 255.

predicted_classes = sport_model.predict(test_X)

for i, img_tagged in enumerate(predicted_classes):
    print(filenames[i], deportes[img_tagged.tolist().index(max(img_tagged))])

